In [56]:
import pandas as pd
import numpy as np

import warnings

from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

## Prepare data
- Display
    Display illustrate the data we are working with 
- X and y meaning
    - X: represents the columns that we need to
    - y: represents the prediction we want the model to target


In [57]:
warnings.filterwarnings('ignore', module='sklearn.preprocessing')

apartments = pd.read_csv("challenge1_train.csv")

display(apartments.head())
print(apartments.columns.tolist())
apartments.info()

X = apartments.drop(columns=["price", "ID", "category"])
y = apartments["price"].copy()

numerical = ["square_feet", "bedrooms", "bathrooms", "latitude", "longitude"]
categorical = ["has_photo", "state"] 

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical)
    ]
)

pipelines = {
    "Linear Regression": Pipeline([
        ("prep", preprocessor),
        ("model", LinearRegression()),
    ]),
    "Decision Tree": Pipeline([
        ("prep", preprocessor),
        ("model", DecisionTreeRegressor(max_depth=15, 
                                        ccp_alpha=8.0, 
                                        max_features=None, 
                                        min_samples_split=2, 
                                        splitter="best", 
                                        min_samples_leaf=8, 
                                        criterion="squared_error", 
                                        random_state=42)),
    ]),
    "Random Forest": Pipeline([
        ("prep", preprocessor),
        ("model", RandomForestRegressor(random_state=42))
    ]),
    "XGBoost": Pipeline([
        ("prep", preprocessor),
        ("model", XGBRegressor(random_state=42))
    ])
}

,ID,price,bathrooms,bedrooms,square_feet,latitude,longitude,category,has_photo,state
0,2,925,1.0,0,116,47.6160,-122.3275,housing/rent/apartment,Thumbnail,WA
1,3,2475,1.0,0,130,40.7629,-73.9885,housing/rent/apartment,Thumbnail,NY
2,5,1695,1.0,0,190,37.7599,-122.4379,housing/rent/apartment,Thumbnail,CA
3,6,1560,1.0,1,200,35.0847,-77.0609,housing/rent/apartment,Thumbnail,NC
4,7,1560,1.0,1,200,35.0960,-77.0272,housing/rent/apartment,Thumbnail,NC


['ID', 'price', 'bathrooms', 'bedrooms', 'square_feet', 'latitude', 'longitude', 'category', 'has_photo', 'state']
<class 'pandas.DataFrame'>
RangeIndex: 7398 entries, 0 to 7397
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   ID           7398 non-null   int64  
 1   price        7398 non-null   int64  
 2   bathrooms    7398 non-null   float64
 3   bedrooms     7398 non-null   int64  
 4   square_feet  7398 non-null   int64  
 5   latitude     7398 non-null   float64
 6   longitude    7398 non-null   float64
 7   category     7398 non-null   str    
 8   has_photo    7398 non-null   str    
 9   state        7398 non-null   str    
dtypes: float64(3), int64(4), str(3)
memory usage: 578.1 KB


### Decision Tree - WIP

In [60]:


param_grid = {
    'model__max_depth': range(14, 20, 2),
    'model__min_samples_split': range(2, 10),
    'model__min_samples_leaf': range(8, 20, 2),
    'model__max_features': [None, 'sqrt', 'log2'],
    'model__max_leaf_nodes': [range(20, 5), None],
    'model__ccp_alpha': [0.0, 0.1, 0.5, 1.0, 2.0, 5.0, 8.0]
}

kf = KFold(n_splits=10, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=pipelines["Decision Tree"],
    param_grid=param_grid,
    cv=kf,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)


print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV RMSE: {-grid_search.best_score_:.3f}")


KeyboardInterrupt: 

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)




pipelines["Decision Tree"].fit(X_train, y_train)
prediction = pipelines["Decision Tree"].predict(X_valid)
scores = cross_val_score(pipelines["Decision Tree"], X_train, y_train, cv=kf, scoring='neg_root_mean_squared_error')


print(f"Decision Tree - RMSE: {np.sqrt(mean_squared_error(y_valid, prediction)):.3f}")
print(f"Decision Tree - CV RMSE: {-scores.mean():.3f}")

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

kf = KFold(n_splits=10, shuffle=True, random_state=42)


pipelines["Decision Tree"].fit(X_train, y_train)
prediction = pipelines["Decision Tree"].predict(X_valid)
print(f"Decision Tree - RMSE: {np.sqrt(mean_squared_error(y_valid, prediction)):.3f}")

## Results

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

for model_name, pipeline in pipelines.items():
    pipeline.fit(X_train, y_train)
    prediction = pipeline.predict(X_valid)
    print(f"{model_name} - RMSE: {np.sqrt(mean_squared_error(y_valid, prediction)):.3f}")

## Visualization
